In [1]:
from dotenv import load_dotenv
import os

load_dotenv()
api_key = os.getenv("ALPHA_VANTAGE_API_KEY")


In [2]:
from alpha_vantage.timeseries import TimeSeries
import pandas as pd

# Load key
from dotenv import load_dotenv
import os
load_dotenv()
api_key = os.getenv("ALPHA_VANTAGE_API_KEY")

# Setup
ts = TimeSeries(key=api_key, output_format='pandas')

# Get data
data, meta = ts.get_daily(symbol='AAPL', outputsize='full')
data = data.rename(columns={
    '1. open': 'open',
    '2. high': 'high',
    '3. low': 'low',
    '4. close': 'close',
    '5. volume': 'volume'
})
data.index = pd.to_datetime(data.index)
data = data.sort_index()
print(data.tail())


               open    high       low   close      volume
date                                                     
2025-06-20  198.235  201.70  196.8550  201.00  96813542.0
2025-06-23  201.625  202.30  198.9600  201.50  55814272.0
2025-06-24  202.590  203.44  200.2000  200.30  54064033.0
2025-06-25  201.450  203.67  200.6201  201.56  39525730.0
2025-06-26  201.430  202.64  199.4600  201.00  50799121.0


In [3]:
import requests

url = "https://www.alphavantage.co/query"
params = {
    "function": "NEWS_SENTIMENT",
    "tickers": "AAPL",
    "apikey": api_key
}

response = requests.get(url, params=params)
news = response.json()

# Display 3 sample headlines
for article in news['feed'][:3]:
    print("📰", article['title'])
    print("🕒", article['time_published'])
    print("🙂", article['overall_sentiment_label'])
    print("🧾", article['summary'])
    print()


📰 Apple's F1 Movie Burns $300 Million-Still Waiting On A Profit - Apple  ( NASDAQ:AAPL ) 
🕒 20250627T145640
🙂 Somewhat-Bullish
🧾 Apple's "F1" racing drama opens in theaters on Friday. The company's video production unit has never turned a profit since its launch in 2017. Geopolitical tensions, Fed uncertainty, and fast-moving headlines are driving July volatility. See how Chris Capre is trading it-live Wednesday, July 2 ...

📰 How To Trade SPY, Top Tech Stocks As Core Inflation Edges Up Above Expectations
🕒 20250627T143536
🙂 Somewhat-Bullish
🧾 Good Morning Traders! In today's Market Clubhouse Morning Memo, we will discuss SPY, QQQ, AAPL, MSFT, NVDA, GOOGL, META, and TSLA. Our proprietary formula, exclusive to Market Clubhouse, dictates these price levels. This dynamic equation takes into account price, volume, and options flow.

📰 Stock Market News for Jun 27, 2025
🕒 20250627T131700
🙂 Neutral
🧾 U.S. stocks closed higher on Thursday, with the S&P 500 inching closer to a new record high 

In [6]:
from collections import defaultdict
import datetime
import pandas as pd

daily_sentiment = defaultdict(list)

for article in news['feed']:
    # Example: "20240627T153000" → split at 'T' → "20240627"
    time_str = article['time_published'].split('T')[0]  # Safe!
    date = datetime.datetime.strptime(time_str, "%Y%m%d").date()
    sentiment = article['overall_sentiment_label']
    daily_sentiment[date].append(sentiment)

# Convert label to score
def label_to_score(label):
    return {'Positive': 1, 'Neutral': 0, 'Negative': -1}.get(label, 0)

# Daily average sentiment
daily_scores = {
    day: sum(label_to_score(s) for s in sentiments) / len(sentiments)
    for day, sentiments in daily_sentiment.items()
}

# Create DataFrame
sentiment_df = pd.DataFrame.from_dict(daily_scores, orient='index', columns=['sentiment_score'])
sentiment_df.index = pd.to_datetime(sentiment_df.index)
print(sentiment_df.tail())


            sentiment_score
2025-06-27              0.0
2025-06-26              0.0
2025-06-25              0.0
2025-06-24              0.0


In [5]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
import numpy as np

# Load FinBERT model (finance-specific sentiment classifier)
model_name = "yiyanghkust/finbert-tone"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

def get_sentiment(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True)
    with torch.no_grad():
        logits = model(**inputs).logits
        probs = torch.nn.functional.softmax(logits, dim=-1)
    labels = ['Negative', 'Neutral', 'Positive']
    return labels[torch.argmax(probs)], probs.squeeze().tolist()

# Example usage:
headline = "Tesla shares surge after earnings beat expectations"
label, scores = get_sentiment(headline)
print(f"Sentiment: {label}, Scores: {scores}")


config.json:   0%|          | 0.00/533 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/439M [00:00<?, ?B/s]

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


Sentiment: Neutral, Scores: [1.1668925381513873e-08, 1.0, 2.495772299937471e-08]


model.safetensors:   0%|          | 0.00/439M [00:00<?, ?B/s]